In [13]:
import sys
import os
import rasterio

sys.path.append(os.path.abspath('..'))

from src.config import load_sentinel2_config
from src.data.sentinel2 import SentinelClient, get_data_profile, get_scene
from src.io import RAW_DIR

In [14]:
RAW_DIR.mkdir(parents=True, exist_ok=True)

cfg = load_sentinel2_config()

client = SentinelClient(cfg)
bbox = cfg.aoi.bounding_box

items = client.search_scenes(bbox)

item = items[0]
assets = item.assets

In [15]:
assets

{'AOT_10m': <Asset href=s3://eodata/Sentinel-2/MSI/L2A_N0500/2021/12/17/S2A_MSIL2A_20211217T111451_N0500_R137_T30STG_20221225T134748.SAFE/GRANULE/L2A_T30STG_A033882_20211217T111454/IMG_DATA/R10m/T30STG_20211217T111451_AOT_10m.jp2>,
 'AOT_20m': <Asset href=s3://eodata/Sentinel-2/MSI/L2A_N0500/2021/12/17/S2A_MSIL2A_20211217T111451_N0500_R137_T30STG_20221225T134748.SAFE/GRANULE/L2A_T30STG_A033882_20211217T111454/IMG_DATA/R20m/T30STG_20211217T111451_AOT_20m.jp2>,
 'AOT_60m': <Asset href=s3://eodata/Sentinel-2/MSI/L2A_N0500/2021/12/17/S2A_MSIL2A_20211217T111451_N0500_R137_T30STG_20221225T134748.SAFE/GRANULE/L2A_T30STG_A033882_20211217T111454/IMG_DATA/R60m/T30STG_20211217T111451_AOT_60m.jp2>,
 'B01_20m': <Asset href=s3://eodata/Sentinel-2/MSI/L2A_N0500/2021/12/17/S2A_MSIL2A_20211217T111451_N0500_R137_T30STG_20221225T134748.SAFE/GRANULE/L2A_T30STG_A033882_20211217T111454/IMG_DATA/R20m/T30STG_20211217T111451_B01_20m.jp2>,
 'B01_60m': <Asset href=s3://eodata/Sentinel-2/MSI/L2A_N0500/2021/12/17/

In [16]:
cfg.msi.bands

{'10m': ['B02_10m', 'B03_10m', 'B04_10m', 'B08_10m'],
 '20m': ['B05_20m',
  'B06_20m',
  'B07_20m',
  'B08A_20m',
  'B11_20m',
  'B12_20m',
  'SCL_20m']}

In [27]:
bands = [band for res in cfg.msi.bands for band in cfg.msi.bands[res]]
bands

['B02_10m',
 'B03_10m',
 'B04_10m',
 'B08_10m',
 'B05_20m',
 'B06_20m',
 'B07_20m',
 'B08A_20m',
 'B11_20m',
 'B12_20m',
 'SCL_20m']

In [36]:
ref_assets = assets[cfg.msi.bands[f"{cfg.msi.target_bands_m}m"][0]].href

In [31]:
profile = get_data_profile(cfg.msi.bands[f"{cfg.msi.target_bands_m}m"][0])

AttributeError: 'str' object has no attribute 'href'

In [ ]:


profile.update(count=11, compress="lzw")

with rasterio.open(RAW_DIR / f"{item.id}_ALLBANDS.tif", "w", **profile) as dst:
    data = get_scene(href)
    dst.write(data, 1)